# Jamaica Mangrove Trend (1996 to 2020) from GMW v3

This notebook uses only the Global Mangrove Watch inputs in:
- `inputs/state_of_art_global_mangrove_layer_bunting_2022/gmw_v3_f1996_t2020_vec`
- `inputs/state_of_art_global_mangrove_layer_bunting_2022/gmw_v3_2020_vec`

Workflow:
1. Build a buffered Jamaica AOI
2. Clip GMW layers to AOI
3. Reproject to EPSG:3448 (Jamaica metric CRS)
4. Compute area-based gain/loss trend statistics (1996 to 2020)
5. Visualize extent and change


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'dphil_papers').exists():
            return p
    raise FileNotFoundError(f'Could not find project root from {start}')

ROOT = find_project_root(Path.cwd())

# Inputs requested by user
gmw_base = ROOT / 'dphil_papers/dphil_paper_3/inputs/state_of_art_global_mangrove_layer_bunting_2022'
gmw_2020_path = gmw_base / 'gmw_v3_2020_vec/gmw_v3_2020_vec.shp'
gmw_change_path = gmw_base / 'gmw_v3_f1996_t2020_vec/gmw_v3_f1996_t2020_vec.shp'

# Jamaica boundary (for AOI)
jamaica_boundary_path = ROOT / 'dphil_papers/dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

# Analysis options
TARGET_CRS = 3448
BUFFER_KM = 10

for p in [gmw_2020_path, gmw_change_path, jamaica_boundary_path]:
    print(p.name, 'exists ->', p.exists())
print('Target CRS EPSG:', TARGET_CRS)
print('Jamaica buffer (km):', BUFFER_KM)


In [ ]:
# Build buffered Jamaica AOI in EPSG:3448, then convert to WGS84 for fast bbox prefilter
jamaica = gpd.read_file(jamaica_boundary_path).to_crs(TARGET_CRS)
aoi = gpd.GeoDataFrame(geometry=jamaica.buffer(BUFFER_KM * 1000), crs=TARGET_CRS)
aoi_wgs84 = aoi.to_crs(4326)

minx, miny, maxx, maxy = aoi_wgs84.total_bounds
print('AOI bbox WGS84:', (round(minx, 4), round(miny, 4), round(maxx, 4), round(maxy, 4)))


In [ ]:
# Read only likely Jamaica features by bbox, then clip exactly to buffered AOI
# (keeps runtime low because global layers are large)

change_bbox = pyogrio.read_dataframe(gmw_change_path, bbox=(minx, miny, maxx, maxy))
extent2020_bbox = pyogrio.read_dataframe(gmw_2020_path, bbox=(minx, miny, maxx, maxy))

change_bbox = gpd.GeoDataFrame(change_bbox, geometry='geometry', crs=4326).to_crs(TARGET_CRS)
extent2020_bbox = gpd.GeoDataFrame(extent2020_bbox, geometry='geometry', crs=4326).to_crs(TARGET_CRS)

change_jm = gpd.overlay(change_bbox, aoi, how='intersection')
extent2020_jm = gpd.overlay(extent2020_bbox, aoi, how='intersection')

change_jm = change_jm[change_jm.geometry.notnull() & ~change_jm.geometry.is_empty].copy()
extent2020_jm = extent2020_jm[extent2020_jm.geometry.notnull() & ~extent2020_jm.geometry.is_empty].copy()

print('Clipped features - change layer:', len(change_jm))
print('Clipped features - 2020 extent:', len(extent2020_jm))
print('CRS change layer:', change_jm.crs)
print('CRS 2020 extent:', extent2020_jm.crs)


In [ ]:
# Inspect change categories present in Jamaica AOI
if 'chng_type' in change_jm.columns:
    category_col = 'chng_type'
elif 'chng_type_' in change_jm.columns:
    category_col = 'chng_type_'
else:
    raise ValueError('Could not find change category field in change layer.')

print('Using change category column:', category_col)
print(change_jm[category_col].value_counts(dropna=False))

if 'PXLVAL' in extent2020_jm.columns:
    print('\n2020 PXLVAL counts:')
    print(extent2020_jm['PXLVAL'].value_counts(dropna=False))


In [ ]:
# Area statistics (hectares) in EPSG:3448
change_jm['area_ha'] = change_jm.geometry.area / 10_000.0
extent2020_jm['area_ha'] = extent2020_jm.geometry.area / 10_000.0

# Normalize category labels
cat = change_jm[category_col].astype(str).str.lower().str.strip()
loss_ha = float(change_jm.loc[cat == 'loss', 'area_ha'].sum())
gain_ha = float(change_jm.loc[cat == 'gain', 'area_ha'].sum())

area_2020_ha = float(extent2020_jm['area_ha'].sum())

# From: 2020 = 1996 - loss + gain  =>  1996 = 2020 + loss - gain
area_1996_est_ha = area_2020_ha + loss_ha - gain_ha
net_change_ha = area_2020_ha - area_1996_est_ha

years = 2020 - 1996

summary = pd.DataFrame([
    {'metric': 'Estimated mangrove area 1996 (ha)', 'value': area_1996_est_ha},
    {'metric': 'Mangrove area 2020 (ha)', 'value': area_2020_ha},
    {'metric': 'Gross loss 1996-2020 (ha)', 'value': loss_ha},
    {'metric': 'Gross gain 1996-2020 (ha)', 'value': gain_ha},
    {'metric': 'Net change 1996-2020 (ha)', 'value': net_change_ha},
    {'metric': 'Net change vs 1996 (%)', 'value': 100.0 * net_change_ha / max(area_1996_est_ha, 1e-9)},
    {'metric': 'Gross loss vs 1996 (%)', 'value': 100.0 * loss_ha / max(area_1996_est_ha, 1e-9)},
    {'metric': 'Gross gain vs 1996 (%)', 'value': 100.0 * gain_ha / max(area_1996_est_ha, 1e-9)},
    {'metric': 'Annual net change (ha/year)', 'value': net_change_ha / years},
])

summary['value'] = summary['value'].astype(float).round(3)
display(summary)


In [ ]:
# Map 1: 2020 mangrove extent in buffered Jamaica AOI
fig, ax = plt.subplots(figsize=(9.5, 8.0), constrained_layout=True)

aoi.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.8)
extent2020_jm.plot(ax=ax, color='#2ca25f', edgecolor='none', alpha=0.85)
jamaica.boundary.plot(ax=ax, color='black', linewidth=1.0, alpha=0.9)

ax.set_title('GMW v3 Mangroves (2020) - Jamaica + Buffer')
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

legend_handles = [
    mpatches.Patch(facecolor='#2ca25f', edgecolor='none', label='Mangrove extent (2020)'),
    mpatches.Patch(facecolor='none', edgecolor='black', label=f'Jamaica + {BUFFER_KM} km buffer'),
]
ax.legend(handles=legend_handles, loc='lower left', frameon=True, fontsize=9)

plt.show()


In [ ]:
# Map 2: Change polygons (gain/loss) 1996-2020 (visibility-enhanced)
# Rendering is enlarged for visual interpretation; statistics remain based on original geometries.

DISPLAY_BUFFER_M = 120   # display-only expansion in meters
ZOOM_PAD_M = 8000        # padding around change bounds

fig, ax = plt.subplots(figsize=(12.5, 10.0), constrained_layout=True)

# Light 2020 extent backdrop for context
extent2020_jm.plot(ax=ax, color='#d9f0d3', edgecolor='none', alpha=0.35)
aoi.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.65)

loss = change_jm[cat == 'loss'].copy()
gain = change_jm[cat == 'gain'].copy()

loss_vis = loss.copy()
gain_vis = gain.copy()
if DISPLAY_BUFFER_M > 0:
    if len(loss_vis) > 0:
        loss_vis['geometry'] = loss_vis.geometry.buffer(DISPLAY_BUFFER_M)
    if len(gain_vis) > 0:
        gain_vis['geometry'] = gain_vis.geometry.buffer(DISPLAY_BUFFER_M)

if len(loss_vis) > 0:
    loss_vis.plot(ax=ax, color='#d73027', edgecolor='#7f0000', linewidth=0.2, alpha=0.90)
if len(gain_vis) > 0:
    gain_vis.plot(ax=ax, color='#1a9850', edgecolor='#00441b', linewidth=0.2, alpha=0.90)

# Overlay true boundaries so exact locations are still visible
if len(loss) > 0:
    loss.boundary.plot(ax=ax, color='#7f0000', linewidth=0.55, alpha=0.9)
if len(gain) > 0:
    gain.boundary.plot(ax=ax, color='#00441b', linewidth=0.55, alpha=0.9)

jamaica.boundary.plot(ax=ax, color='black', linewidth=1.0, alpha=0.85)

# Zoom to where change exists so features appear larger
if len(change_jm) > 0:
    minx, miny, maxx, maxy = change_jm.total_bounds
    ax.set_xlim(minx - ZOOM_PAD_M, maxx + ZOOM_PAD_M)
    ax.set_ylim(miny - ZOOM_PAD_M, maxy + ZOOM_PAD_M)

ax.set_title(
    f'GMW v3 Mangrove Change (1996-2020) - Jamaica + Buffer\n'
    f'Visibility-enhanced rendering (display buffer = {DISPLAY_BUFFER_M} m)',
    fontsize=12,
)
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

legend_handles = [
    mpatches.Patch(facecolor='#d73027', edgecolor='none', label='Loss'),
    mpatches.Patch(facecolor='#1a9850', edgecolor='none', label='Gain'),
    mpatches.Patch(facecolor='#d9f0d3', edgecolor='none', label='2020 mangrove extent (context)'),
]
ax.legend(handles=legend_handles, loc='lower left', frameon=True, fontsize=9)

plt.show()


In [ ]:
# Chart: Gross loss/gain and net trend
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), constrained_layout=True)

axes[0].bar(['Loss', 'Gain'], [loss_ha, gain_ha], color=['#d73027', '#1a9850'])
axes[0].set_ylabel('Area (ha)')
axes[0].set_title('Gross Change (1996-2020)')

axes[1].bar(['1996 est.', '2020'], [area_1996_est_ha, area_2020_ha], color=['#9ecae1', '#2ca25f'])
axes[1].set_ylabel('Area (ha)')
axes[1].set_title('Estimated Net Trend')

for ax in axes:
    for p in ax.patches:
        h = p.get_height()
        ax.annotate(f'{h:,.0f}', (p.get_x() + p.get_width()/2, h),
                    ha='center', va='bottom', fontsize=9, xytext=(0, 3), textcoords='offset points')

plt.show()


## Interpretation Notes
- `loss` and `gain` are gross changes from 1996 to 2020.
- Net trend is estimated from the relationship: `Area_2020 = Area_1996 - Loss + Gain`.
- Results are for **Jamaica + buffer AOI** (set by `BUFFER_KM`).
- Change polygons show where transitions occurred; they do not by themselves represent current 2020 extent.
